# Notebook 03 — Gold · Conciliação e Aging

**Objetivo:** Aplicar a lógica de negócio da conciliação — calcular saldo, classificar status e segmentar pendências e divergências por faixa de aging.  
**Fonte:** Schema `silver`  
**Destino:** Schema `gold`  
**Autor:** Davi Alves  
**Data de referência:** 2025-05-31 (último dia do mês mais recente dos dados)

### Regras de negócio
| Status | Condição | Saldo |
|---|---|---|
| Conciliado | Valor_Realizado = Valor_Previsto | 0 |
| Divergente | Valor_Realizado ≠ Valor_Previsto | Valor_Previsto - Valor_Realizado |
| Pendente | Valor_Realizado = nulo | Valor_Previsto |

### Faixas de aging — Pendentes
Calculado a partir de `Data_Prevista`. Apenas lançamentos com `Data_Prevista <= data_referencia` entram no aging.

| Ordem | Faixa | Dias em aberto | Criticidade |
|---|---|---|---|
| 1 | 0 a 30 dias | <= 30 | Normal |
| 2 | 31 a 60 dias | <= 60 | Atenção |
| 3 | 61 a 90 dias | <= 90 | Elevado |
| 4 | 91 a 120 dias | <= 120 | Alto |
| 5 | 121 a 180 dias | <= 180 | Crítico |
| 6 | 181 a 360 dias | <= 360 | Muito Crítico |
| 7 | +360 dias | > 360 | Máximo |

### Faixas de aging — Divergentes
Calculado a partir de `Data_Liquidacao`. Mede há quantos dias o pagamento foi feito com valor divergente e ainda não foi corrigido.

| Ordem | Faixa | Dias desde liquidação | Criticidade |
|---|---|---|---|
| 1 | 0 a 30 dias | <= 30 | Normal |
| 2 | 31 a 60 dias | <= 60 | Atenção |
| 3 | 61 a 90 dias | <= 90 | Elevado |
| 4 | 91 a 120 dias | <= 120 | Alto |
| 5 | 121 a 180 dias | <= 180 | Crítico |
| 6 | 181 a 360 dias | <= 360 | Muito Crítico |
| 7 | +360 dias | > 360 | Máximo |

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

df_silver = spark.table("workspace.silver.transacoes")

print("Schema gold criado com sucesso")
print(f"Silver carregado — {df_silver.count()} linhas")

Schema gold criado com sucesso
Silver carregado — 3030 linhas


In [0]:
from pyspark.sql.functions import col, when, datediff, lit
from pyspark.sql.types import DecimalType

data_referencia = lit("2025-05-31").cast("date")

df_gold = df_silver \
    .withColumn("Saldo",
        when(col("Status") == "Conciliado", lit(0).cast(DecimalType(15,2)))
        .when(col("Status") == "Divergente", col("Valor_Previsto") - col("Valor_Realizado"))
        .when(col("Status") == "Pendente",   col("Valor_Previsto"))
    ) \
    .withColumn("Dias_em_Aberto",
        when(
            (col("Status") == "Pendente") &
            (col("Data_Prevista") <= data_referencia),
            datediff(data_referencia, col("Data_Prevista")))
        .otherwise(lit(None))
    ) \
    .withColumn("Dias_Divergencia",
        when(
            (col("Status") == "Divergente") &
            (col("Data_Liquidacao") <= data_referencia),
            datediff(data_referencia, col("Data_Liquidacao")))
        .otherwise(lit(None))
    ) \
    .withColumn("Faixa_Aging",
        when(col("Status") != "Pendente",        lit(None))
        .when(col("Dias_em_Aberto").isNull(),     lit(None))
        .when(col("Dias_em_Aberto") <= 30,        lit("0 a 30 dias"))
        .when(col("Dias_em_Aberto") <= 60,        lit("31 a 60 dias"))
        .when(col("Dias_em_Aberto") <= 90,        lit("61 a 90 dias"))
        .when(col("Dias_em_Aberto") <= 120,       lit("91 a 120 dias"))
        .when(col("Dias_em_Aberto") <= 180,       lit("121 a 180 dias"))
        .when(col("Dias_em_Aberto") <= 360,       lit("181 a 360 dias"))
        .otherwise(lit("+360 dias"))
    ) \
    .withColumn("Faixa_Aging_Div",
        when(col("Status") != "Divergente",       lit(None))
        .when(col("Dias_Divergencia").isNull(),    lit(None))
        .when(col("Dias_Divergencia") <= 30,       lit("0 a 30 dias"))
        .when(col("Dias_Divergencia") <= 60,       lit("31 a 60 dias"))
        .when(col("Dias_Divergencia") <= 90,       lit("61 a 90 dias"))
        .when(col("Dias_Divergencia") <= 120,      lit("91 a 120 dias"))
        .when(col("Dias_Divergencia") <= 180,      lit("121 a 180 dias"))
        .when(col("Dias_Divergencia") <= 360,      lit("181 a 360 dias"))
        .otherwise(lit("+360 dias"))
    ) \
    .withColumn("Criticidade",
        when(col("Status") != "Pendente",        lit(None))
        .when(col("Dias_em_Aberto").isNull(),     lit(None))
        .when(col("Dias_em_Aberto") <= 30,        lit("Normal"))
        .when(col("Dias_em_Aberto") <= 60,        lit("Atenção"))
        .when(col("Dias_em_Aberto") <= 90,        lit("Elevado"))
        .when(col("Dias_em_Aberto") <= 120,       lit("Alto"))
        .when(col("Dias_em_Aberto") <= 180,       lit("Crítico"))
        .when(col("Dias_em_Aberto") <= 360,       lit("Muito Crítico"))
        .otherwise(lit("Máximo"))
    ) \
    .withColumn("Criticidade_Div",
        when(col("Status") != "Divergente",       lit(None))
        .when(col("Dias_Divergencia").isNull(),    lit(None))
        .when(col("Dias_Divergencia") <= 30,       lit("Normal"))
        .when(col("Dias_Divergencia") <= 60,       lit("Atenção"))
        .when(col("Dias_Divergencia") <= 90,       lit("Elevado"))
        .when(col("Dias_Divergencia") <= 120,      lit("Alto"))
        .when(col("Dias_Divergencia") <= 180,      lit("Crítico"))
        .when(col("Dias_Divergencia") <= 360,      lit("Muito Crítico"))
        .otherwise(lit("Máximo"))
    ) \
    .withColumn("Ordem_Aging",
        when(col("Status") != "Pendente",             lit(None))
        .when(col("Faixa_Aging") == "0 a 30 dias",    lit(1))
        .when(col("Faixa_Aging") == "31 a 60 dias",   lit(2))
        .when(col("Faixa_Aging") == "61 a 90 dias",   lit(3))
        .when(col("Faixa_Aging") == "91 a 120 dias",  lit(4))
        .when(col("Faixa_Aging") == "121 a 180 dias", lit(5))
        .when(col("Faixa_Aging") == "181 a 360 dias", lit(6))
        .when(col("Faixa_Aging") == "+360 dias",       lit(7))
    ) \
    .withColumn("Ordem_Aging_Div",
        when(col("Status") != "Divergente",               lit(None))
        .when(col("Faixa_Aging_Div") == "0 a 30 dias",    lit(1))
        .when(col("Faixa_Aging_Div") == "31 a 60 dias",   lit(2))
        .when(col("Faixa_Aging_Div") == "61 a 90 dias",   lit(3))
        .when(col("Faixa_Aging_Div") == "91 a 120 dias",  lit(4))
        .when(col("Faixa_Aging_Div") == "121 a 180 dias", lit(5))
        .when(col("Faixa_Aging_Div") == "181 a 360 dias", lit(6))
        .when(col("Faixa_Aging_Div") == "+360 dias",       lit(7))
    )

print("Lógica aplicada")
print(f"Linhas: {df_gold.count()}")

print("\n=== Aging Pendentes ===")
df_gold.filter(col("Status") == "Pendente") \
       .groupBy("Ordem_Aging", "Faixa_Aging", "Criticidade").count() \
       .orderBy("Ordem_Aging").show()

print("\n=== Aging Divergentes ===")
df_gold.filter(col("Status") == "Divergente") \
       .groupBy("Ordem_Aging_Div", "Faixa_Aging_Div", "Criticidade_Div").count() \
       .orderBy("Ordem_Aging_Div").show()

Lógica aplicada
Linhas: 3030

=== Aging Pendentes ===
+-----------+--------------+-------------+-----+
|Ordem_Aging|   Faixa_Aging|  Criticidade|count|
+-----------+--------------+-------------+-----+
|          1|   0 a 30 dias|       Normal|    3|
|          2|  31 a 60 dias|      Atenção|   23|
|          3|  61 a 90 dias|      Elevado|   15|
|          4| 91 a 120 dias|         Alto|   17|
|          5|121 a 180 dias|      Crítico|   37|
|          6|181 a 360 dias|Muito Crítico|  124|
|          7|     +360 dias|       Máximo|  102|
+-----------+--------------+-------------+-----+


=== Aging Divergentes ===
+---------------+---------------+---------------+-----+
|Ordem_Aging_Div|Faixa_Aging_Div|Criticidade_Div|count|
+---------------+---------------+---------------+-----+
|              1|    0 a 30 dias|         Normal|    4|
|              2|   31 a 60 dias|        Atenção|   20|
|              3|   61 a 90 dias|        Elevado|   15|
|              4|  91 a 120 dias|          

In [0]:
print("=== Distribuição por Status ===")
df_gold.groupBy("Status").count().orderBy("Status").show()

print("=== Saldo total por Status ===")
from pyspark.sql.functions import sum as spark_sum, round as spark_round
df_gold.groupBy("Status") \
       .agg(spark_round(spark_sum("Saldo"), 2).alias("Saldo_Total")) \
       .orderBy("Status").show()

print("=== Aging Pendentes ===")
df_gold.filter(col("Status") == "Pendente") \
       .groupBy("Ordem_Aging", "Faixa_Aging", "Criticidade").count() \
       .orderBy("Ordem_Aging").show()

print("=== Aging Divergentes ===")
df_gold.filter(col("Status") == "Divergente") \
       .groupBy("Ordem_Aging_Div", "Faixa_Aging_Div", "Criticidade_Div").count() \
       .orderBy("Ordem_Aging_Div").show()

=== Distribuição por Status ===
+----------+-----+
|    Status|count|
+----------+-----+
|Conciliado| 2395|
|Divergente|  314|
|  Pendente|  321|
+----------+-----+

=== Saldo total por Status ===
+----------+-----------+
|    Status|Saldo_Total|
+----------+-----------+
|Conciliado|       0.00|
|Divergente|  606828.04|
|  Pendente| 8032226.15|
+----------+-----------+

=== Aging Pendentes ===
+-----------+--------------+-------------+-----+
|Ordem_Aging|   Faixa_Aging|  Criticidade|count|
+-----------+--------------+-------------+-----+
|          1|   0 a 30 dias|       Normal|    3|
|          2|  31 a 60 dias|      Atenção|   23|
|          3|  61 a 90 dias|      Elevado|   15|
|          4| 91 a 120 dias|         Alto|   17|
|          5|121 a 180 dias|      Crítico|   37|
|          6|181 a 360 dias|Muito Crítico|  124|
|          7|     +360 dias|       Máximo|  102|
+-----------+--------------+-------------+-----+

=== Aging Divergentes ===
+---------------+---------------+----

In [0]:
# Salva Gold atualizado com overwriteSchema para aceitar novas colunas
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.conciliacao")

print("Gold salvo com sucesso")
print(f"gold.conciliacao: {spark.table('workspace.gold.conciliacao').count()} linhas")
print("\nColunas da tabela Gold:")
spark.table("workspace.gold.conciliacao").printSchema()

Gold salvo com sucesso
gold.conciliacao: 3030 linhas

Colunas da tabela Gold:
root
 |-- ID_Transacao: string (nullable = true)
 |-- Data_Lancamento: date (nullable = true)
 |-- Data_Prevista: date (nullable = true)
 |-- Data_Liquidacao: date (nullable = true)
 |-- Valor_Previsto: decimal(15,2) (nullable = true)
 |-- Valor_Realizado: decimal(15,2) (nullable = true)
 |-- Status: string (nullable = true)
 |-- Tipo_Lancamento: string (nullable = true)
 |-- ID_Conta: string (nullable = true)
 |-- ID_Fornecedor: string (nullable = true)
 |-- Descricao: string (nullable = true)
 |-- Canal: string (nullable = true)
 |-- Saldo: decimal(16,2) (nullable = true)
 |-- Dias_em_Aberto: integer (nullable = true)
 |-- Dias_Divergencia: integer (nullable = true)
 |-- Faixa_Aging: string (nullable = true)
 |-- Faixa_Aging_Div: string (nullable = true)
 |-- Criticidade: string (nullable = true)
 |-- Criticidade_Div: string (nullable = true)
 |-- Ordem_Aging: integer (nullable = true)
 |-- Ordem_Aging_Div:

## Relatório final — Gold

### Tabela gold.conciliacao
| Coluna | Tipo | Origem |
|---|---|---|
| ID_Transacao | string | Silver |
| Data_Lancamento | date | Silver |
| Data_Prevista | date | Silver |
| Data_Liquidacao | date | Silver |
| Valor_Previsto | decimal(15,2) | Silver |
| Valor_Realizado | decimal(15,2) | Silver |
| Status | string | Silver |
| Tipo_Lancamento | string | Silver |
| ID_Conta | string | Silver |
| ID_Fornecedor | string | Silver |
| Descricao | string | Silver |
| Canal | string | Silver |
| Saldo | decimal(16,2) | **Calculado** |
| Dias_em_Aberto | integer | **Calculado — Pendentes** |
| Dias_Divergencia | integer | **Calculado — Divergentes** |
| Faixa_Aging | string | **Calculado — Pendentes** |
| Faixa_Aging_Div | string | **Calculado — Divergentes** |
| Criticidade | string | **Calculado — Pendentes** |
| Criticidade_Div | string | **Calculado — Divergentes** |
| Ordem_Aging | integer | **Calculado — Pendentes** |
| Ordem_Aging_Div | integer | **Calculado — Divergentes** |

### Resultado da conciliação
| Status | Qtd | Saldo total |
|---|---|---|
| Conciliado | 2.395 | R$ 0,00 |
| Divergente | 314 | R$ 606.828,04 |
| Pendente | 321 | R$ 8.032.226,15 |

### Aging de Pendentes
Data de referência: **2025-05-31**. Calculado a partir de `Data_Prevista`.  
Apenas lançamentos com `Data_Prevista <= 2025-05-31` entram no aging.

| Ordem | Faixa | Qtd |
|---|---|---|
| 1 | 0 a 30 dias | 3 |
| 2 | 31 a 60 dias | 23 |
| 3 | 61 a 90 dias | 15 |
| 4 | 91 a 120 dias | 17 |
| 5 | 121 a 180 dias | 37 |
| 6 | 181 a 360 dias | 124 |
| 7 | +360 dias | 102 |

### Aging de Divergentes
Data de referência: **2025-05-31**. Calculado a partir de `Data_Liquidacao`.  
Mede há quantos dias o pagamento foi feito com valor divergente e ainda não foi corrigido.

| Ordem | Faixa | Qtd |
|---|---|---|
| 1 | 0 a 30 dias | 4 |
| 2 | 31 a 60 dias | 20 |
| 3 | 61 a 90 dias | 15 |
| 4 | 91 a 120 dias | 20 |
| 5 | 121 a 180 dias | 37 |
| 6 | 181 a 360 dias | 121 |
| 7 | +360 dias | 97 |

### Próximo passo
Exportar `gold.conciliacao` como CSV e importar no Power BI para construção do dashboard.

In [0]:
# Exporta Gold atualizado como CSV para o Volume
df_gold.coalesce(1).write \
    .option("header", True) \
    .mode("overwrite") \
    .csv("/Volumes/workspace/conciliacao/raw_data/gold_conciliacao")

print("CSV exportado com sucesso")

CSV exportado com sucesso
